In [1]:
#@title 1) Mount Google Drive + Imports
# Hybrid Swiss LTr audit script
# - Same input files as your previous script
# - Only creates ONE multi-month output workbook
# - Uses hybrid logic:
#   * service_date = shift/session start date for streak/rest/span/break checks
#   * calendar_date = actual physical date where minutes occurred for 50h/week and month allocation

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import re
import numpy as np
import pandas as pd
import pytz

from datetime import datetime, timedelta, time as dtime
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

SWISS_TZ = pytz.timezone("Europe/Zurich")

print("✅ Drive mounted. TZ =", SWISS_TZ)

Mounted at /content/drive
✅ Drive mounted. TZ = Europe/Zurich


In [2]:
#@title 2) SETTINGS — Upload inputs & set output folder

from google.colab import files
import os

print("=== UPLOAD MATCHED COLLABS FILE ===")
uploaded_collabs = files.upload()
MATCHED_COLLABS_PATH = list(uploaded_collabs.keys())[0]

print("\n=== UPLOAD MERGED RDA FILE ===")
uploaded_rda = files.upload()
MERGED_RDA_PATH = list(uploaded_rda.keys())[0]

print("\n=== OUTPUT FOLDER ===")
folder_name = input("Enter the name of the output folder (will be saved in Drive): ")

# --- Output folder ---
MONTHLY_OUTPUT_FOLDER = f"/content/drive/MyDrive/{folder_name}"
MULTIPLE_OUTPUT_FOLDER = os.path.join(MONTHLY_OUTPUT_FOLDER, "multiple")
MULTIPLE_BOOK_PATH = os.path.join(MULTIPLE_OUTPUT_FOLDER, "LTR_CHECKS_MULTIPLE_ALL_MONTHS_HYBRID.xlsx")

os.makedirs(MULTIPLE_OUTPUT_FOLDER, exist_ok=True)

# --- Interval sanity ---
MAX_SINGLE_INTERVAL_HOURS = 24

# --- Hybrid service logic ---
# A new service/shift starts after a true daily rest gap.
# This prevents a previous day's overnight tail + early morning pause from creating a fake new worked day.
SERVICE_RESET_GAP_HOURS = 8.0

# --- Rule thresholds ---
WEEKLY_LIMIT_HOURS = 50.0
SPAN_LIMIT_HOURS = 14.0
REST_NORMAL_HOURS = 11.0
REST_ABSOLUTE_MIN_HOURS = 8.0

# If True, a reduced rest from 8h to <11h is only accepted when the 14-day backward average is known and >= 11h.
# If the file starts and lacks enough history, first reduced rests become REVIEW, not hard infractions.
REQUIRE_AVG_FOR_REDUCED_REST = True

# --- Flags ---
RUN_OVER50H = True
RUN_STREAK  = True
RUN_SPAN    = True
RUN_REST11  = True
RUN_BREAKS  = True

# --- Business toggles ---
COUNT_TRANSPORT_AS_WORK_FOR_50H    = True
COUNT_TRANSPORT_AS_WORK_FOR_STREAK = True
COUNT_TRANSPORT_AS_WORK_FOR_BREAKS = True
COUNT_TRANSPORT_FOR_SERVICE_BOUNDARY = True

# --- Codes ---
PAUSE_CODES = {"16009", "95900"}
TRANSPORT_CODES = {"61800", "61010"}
EXCLUDE_PRESTATIONS = {"196", "60041"}
PSEUDO_NON_WORK_CODES = {"195"}

print("\n✅ SETTINGS OK")
print("Matched collabs :", MATCHED_COLLABS_PATH)
print("Merged RDA      :", MERGED_RDA_PATH)
print("Output workbook :", MULTIPLE_BOOK_PATH)
print("Service reset gap:", SERVICE_RESET_GAP_HOURS, "hours")

=== UPLOAD MATCHED COLLABS FILE ===


Saving mapping_collabs&clients&wf.xlsx to mapping_collabs&clients&wf.xlsx

=== UPLOAD MERGED RDA FILE ===


Saving merged_2025_herewego_20260512_100821.xlsx to merged_2025_herewego_20260512_100821.xlsx

=== OUTPUT FOLDER ===
Enter the name of the output folder (will be saved in Drive): ltr check all UO

✅ SETTINGS OK
Matched collabs : mapping_collabs&clients&wf.xlsx
Merged RDA      : merged_2025_herewego_20260512_100821.xlsx
Output workbook : /content/drive/MyDrive/ltr check all UO/multiple/LTR_CHECKS_MULTIPLE_ALL_MONTHS_HYBRID.xlsx
Service reset gap: 8.0 hours


In [3]:
#@title 3) Swiss-FR formatting helpers + export cleanup

SWISS_DATE_FMT     = "DD.MM.YYYY"
SWISS_DATETIME_FMT = "DD.MM.YYYY HH:MM"
NUM_INT_FMT        = "0"
NUM_1D_FMT         = "0.0"
NUM_2D_FMT         = "0.00"

VALID_INTERVAL_STATUSES = {"OK", "OK_OVERNIGHT"}

STATUS_ORDER = {
    "OK_BRANCH": 0,
    "OK_FALLBACK": 1,
    "UNIQUE_CODE_NO_BRANCH": 2,
    "AMBIG_MATCH": 3,
    "MISSING_CODE": 4,
}


def _clean_str(x):
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s.lower() in {"", "nan", "none", "nat"} else s


def _clean_name_key(x):
    s = _clean_str(x).lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9àâäçéèêëîïôöùûüÿñæœ ]+", "", s)
    return s.strip()


def _series_or_default(df: pd.DataFrame, col: str, default=""):
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index, dtype=object)


def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df
    if not df.columns.duplicated().any():
        return df

    df = df.copy()
    uniq_names = list(dict.fromkeys(df.columns))
    pieces = []

    for name in uniq_names:
        idxs = [i for i, c in enumerate(df.columns) if c == name]
        if len(idxs) == 1:
            pieces.append(df.iloc[:, idxs[0]])
        else:
            block = df.iloc[:, idxs].copy().replace({"": np.nan})
            combined = block.bfill(axis=1).iloc[:, 0]
            pieces.append(combined)

    out = pd.concat(pieces, axis=1)
    out.columns = uniq_names
    return out


def reorder_columns(df: pd.DataFrame, preferred: list) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df
    first = [c for c in preferred if c in df.columns]
    rest  = [c for c in df.columns if c not in first]
    return df[first + rest]


def excel_safe_no_tz(df: pd.DataFrame) -> pd.DataFrame:
    if df is None:
        return pd.DataFrame()
    df = ensure_unique_columns(df.copy())
    for col in df.columns:
        s = df[col]
        if isinstance(s.dtype, pd.DatetimeTZDtype):
            df[col] = s.dt.tz_convert(SWISS_TZ).dt.tz_localize(None)
        elif s.dtype == "object":
            def _strip_tz(x):
                if isinstance(x, pd.Timestamp) and x.tzinfo is not None:
                    return x.tz_convert(SWISS_TZ).tz_localize(None)
                return x
            df[col] = s.map(_strip_tz)
    return df


def prep_for_export(df: pd.DataFrame, drop_collaborateur_id=True) -> pd.DataFrame:
    if df is None:
        return pd.DataFrame()
    df = ensure_unique_columns(df.copy())
    if df.empty:
        return df

    uid = pd.Series([""] * len(df), index=df.index, dtype=object)
    for c in ["collab_uid", "collab_key", "collab_master_id"]:
        if c in df.columns:
            fallback = df[c].astype(str).map(_clean_str)
            uid = uid.where(uid != "", fallback)

    if (uid == "").any():
        code_fallback = _series_or_default(df, "No_collaborateur_codes", "").astype(str).map(_clean_str)
        name_fallback = _series_or_default(df, "Collaborateur", "").astype(str).map(_clean_str)
        uid = uid.where(uid != "", ("UNMATCHED::" + code_fallback + "::" + name_fallback).map(_clean_str))

    df["collab_uid"] = uid

    if drop_collaborateur_id and "Collaborateur_ID" in df.columns:
        df = df.drop(columns=["Collaborateur_ID"])

    return ensure_unique_columns(df)


def _auto_width(ws, max_w=65, min_w=10):
    for col in range(1, ws.max_column + 1):
        letter = get_column_letter(col)
        max_len = 0
        for cell in ws[letter]:
            v = cell.value
            if v is None:
                continue
            max_len = max(max_len, len(str(v)))
        ws.column_dimensions[letter].width = max(min_w, min(max_w, max_len + 2))


def apply_swiss_formats_xlsx(xlsx_path: str):
    wb = load_workbook(xlsx_path)
    wrap = Alignment(wrap_text=True, vertical="top")

    def fmt_from_header(h):
        h_raw = str(h or "").strip()
        h = h_raw.lower()

        datetime_keys = [
            "start", "end", "début", "fin", "session_start", "session_end",
            "service_start", "service_end", "previous_service_end",
            "current_service_start", "slice_start", "slice_end", "timestamp"
        ]
        if any(k in h for k in datetime_keys) and not any(k in h for k in ["week", "semaine"]):
            if "date" not in h or "datetime" in h or "début" in h or "fin" in h or "start" in h or "end" in h:
                return SWISS_DATETIME_FMT

        if any(k in h for k in ["minutes", "(min)", "_min"]):
            return NUM_INT_FMT
        if any(k in h for k in ["heures", "hours", "repos", "rest", "amplitude"]):
            return NUM_2D_FMT

        date_headers = {
            "date", "jour", "event_date", "target_month", "mois", "service_date", "calendar_date",
            "week_monday", "week_sunday", "début semaine", "fin semaine",
            "début de semaine", "fin de semaine"
        }
        if h in date_headers or h.endswith("_date"):
            return SWISS_DATE_FMT

        return None

    for sheet in wb.sheetnames:
        ws = wb[sheet]
        ws.freeze_panes = "A2"
        for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
            for cell in row:
                cell.alignment = wrap

        headers = [ws.cell(row=1, column=c).value for c in range(1, ws.max_column + 1)]
        for c, header in enumerate(headers, start=1):
            fmt = fmt_from_header(header)
            if fmt is None:
                continue
            for r in range(2, ws.max_row + 1):
                cell = ws.cell(row=r, column=c)
                if cell.value is None or cell.value == "":
                    continue
                cell.number_format = fmt
        _auto_width(ws)
    wb.save(xlsx_path)
    print("✅ Swiss-FR formats applied:", xlsx_path)

In [4]:
#@title 4) Parsing + normalization + collaborator mapping

def pick_worst_status(values):
    vals = [_clean_str(v) for v in values if _clean_str(v) != ""]
    if not vals:
        return ""
    return max(vals, key=lambda x: STATUS_ORDER.get(x, -1))


def _norm_id(x):
    if x is None:
        return ""
    try:
        if isinstance(x, float) and np.isnan(x):
            return ""
    except Exception:
        pass
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "nat"}:
        return ""
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    if re.fullmatch(r"\d+", s):
        return s
    digits = re.sub(r"\D+", "", s)
    return digits if digits else s


def _parse_time_series(series):
    t = pd.to_datetime(series, format="%H:%M:%S", errors="coerce")
    miss = t.isna()
    if miss.any():
        t2 = pd.to_datetime(series[miss], format="%H:%M", errors="coerce")
        t.loc[miss] = t2
    return t.dt.time


def _parse_datetime_robust(s: pd.Series) -> pd.Series:
    s = s.copy()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    if s.dtype == object:
        s = s.replace({"": np.nan, "NaN": np.nan, "nan": np.nan, "None": np.nan})

    m_dt = s.apply(lambda x: isinstance(x, (pd.Timestamp, datetime)))
    if m_dt.any():
        out.loc[m_dt] = pd.to_datetime(s.loc[m_dt], errors="coerce")

    def _is_num(x):
        if x is None:
            return False
        try:
            if isinstance(x, float) and np.isnan(x):
                return False
        except Exception:
            pass
        if isinstance(x, (int, float, np.integer, np.floating)):
            return True
        if isinstance(x, str):
            try:
                float(x.strip())
                return True
            except Exception:
                return False
        return False

    m_num = s.apply(_is_num) & (~m_dt)
    if m_num.any():
        num = pd.to_numeric(s.loc[m_num], errors="coerce")
        ok = num.notna()
        if ok.any():
            out.loc[num.loc[ok].index] = pd.to_datetime(num.loc[ok], unit="d", origin="1899-12-30", errors="coerce")

    m_str = s.apply(lambda x: isinstance(x, str)) & (~m_dt) & (~m_num)
    if m_str.any():
        s_str = s.loc[m_str].astype(str).str.strip()
        parsed = pd.to_datetime(s_str, errors="coerce", dayfirst=True)
        remain = parsed.isna()
        if remain.any():
            fmts = [
                "%d.%m.%Y %H:%M:%S", "%d.%m.%Y %H:%M",
                "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M",
                "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M",
                "%d.%m.%Y", "%d/%m/%Y", "%Y-%m-%d",
            ]
            fixed = parsed.copy()
            for fmt in fmts:
                need = fixed.isna()
                if not need.any():
                    break
                fixed.loc[need] = pd.to_datetime(s_str.loc[need], format=fmt, errors="coerce")
            parsed = fixed
        out.loc[m_str] = parsed
    return out


def _localize_date_series_naive(series):
    # We keep datetimes naive for Excel safety, but Date début is retained as a datetime-like day value.
    return pd.to_datetime(pd.to_datetime(series, errors="coerce").dt.date, errors="coerce")


def load_and_normalize(path: str) -> pd.DataFrame:
    lower = str(path).lower()

    if lower.endswith((".xlsx", ".xls")):
        df = pd.read_excel(path, sheet_name=0, dtype=object)
        df.columns = df.columns.str.strip().str.lstrip("\ufeff")
        if not {"Début", "Fin"}.issubset(df.columns):
            raise ValueError("Excel must contain 'Début' and 'Fin' columns.")
    else:
        df = pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=object, low_memory=False)
        if df.shape[1] <= 2:
            df = pd.read_csv(path, sep=",", encoding="utf-8-sig", dtype=object, low_memory=False)
        df.columns = df.columns.str.strip().str.lstrip("\ufeff")

    for c in ["Collaborateur", "No collaborateur", "Prestation", "No prestation", "Durée"]:
        if c not in df.columns:
            df[c] = np.nan

    if {"Début", "Fin"}.issubset(df.columns):
        df["Début"] = _parse_datetime_robust(df["Début"])
        df["Fin"]   = _parse_datetime_robust(df["Fin"])
        df["Heure début"] = pd.to_datetime(df["Début"], errors="coerce").dt.time
        df["Heure fin"]   = pd.to_datetime(df["Fin"], errors="coerce").dt.time
        df["Date début"]  = _localize_date_series_naive(df["Début"])
    else:
        needed = {"Date début", "Heure début", "Heure fin"}
        if not needed.issubset(df.columns):
            raise ValueError(f"CSV must contain {sorted(list(needed))} OR Début/Fin.")
        df["Date début"] = pd.to_datetime(df["Date début"], format="%d.%m.%Y", errors="coerce")
        df["Heure début"] = _parse_time_series(df["Heure début"])
        df["Heure fin"]   = _parse_time_series(df["Heure fin"])
        if "Début" not in df.columns:
            df["Début"] = pd.NaT
        if "Fin" not in df.columns:
            df["Fin"] = pd.NaT

    df["No prestation"] = df["No prestation"].astype(str).str.strip()
    df["Prestation"] = df["Prestation"].astype(str).str.strip()
    df["No collaborateur"] = df["No collaborateur"].apply(_norm_id)
    df["Durée"] = pd.to_numeric(df["Durée"], errors="coerce").fillna(0)

    before = len(df)
    df = df[~df["No prestation"].isin(EXCLUDE_PRESTATIONS)].copy()
    print(f"✅ Excluded prestations {EXCLUDE_PRESTATIONS}: {before} -> {len(df)} rows")

    df["Mois"] = pd.to_datetime(df["Date début"], errors="coerce").dt.strftime("%Y-%m")
    return ensure_unique_columns(df)


def load_matched_collabs(path: str) -> pd.DataFrame:
    m = pd.read_excel(path, sheet_name="Matched Collaborateurs", dtype=object)
    m.columns = m.columns.str.strip().str.lstrip("\ufeff")

    required = {
        "collaborateur-id",
        "name-collaborateur",
        "no-collaborateur-sa-101",
        "no-collaborateur-sarl-102",
        "no-collaborateur-ne-103",
    }
    miss = [c for c in required if c not in m.columns]
    if miss:
        raise ValueError(f"Matched Collaborateurs sheet missing columns: {miss}")

    for c in ["no-collaborateur-sa-101", "no-collaborateur-sarl-102", "no-collaborateur-ne-103"]:
        m[c] = m[c].apply(_norm_id)

    m["collaborateur-id"] = m["collaborateur-id"].astype(str).str.strip()
    m["name-collaborateur"] = m["name-collaborateur"].astype(str).str.strip()
    return m


def _find_oe_column(cols):
    for c in cols:
        if str(c).strip().lower() == "oe":
            return str(c)
    for c in cols:
        cL = str(c).lower()
        if "oe" in cL and any(k in cL for k in ["complet", "code", "structure", "organisation", "orga"]):
            return str(c)
    return None


def _branch_hint_from_oe(oe_val) -> str:
    s = _norm_id(oe_val)
    if not s:
        return ""
    last3 = s[-3:] if len(s) >= 3 else s
    if last3 == "101":
        return "sa"
    if last3 in {"102", "201"}:
        return "sarl"
    if last3 in {"103", "301"}:
        return "ne"
    return ""


def _make_match_long(matched: pd.DataFrame) -> pd.DataFrame:
    branch_map = {
        "sa":   "no-collaborateur-sa-101",
        "sarl": "no-collaborateur-sarl-102",
        "ne":   "no-collaborateur-ne-103",
    }
    parts = []
    for br, col in branch_map.items():
        tmp = matched[["collaborateur-id", "name-collaborateur", col]].copy()
        tmp = tmp.rename(columns={
            "collaborateur-id": "collab_master_id",
            "name-collaborateur": "collab_display_name",
            col: "no_collab_norm",
        })
        tmp["branch"] = br
        tmp["no_collab_norm"] = tmp["no_collab_norm"].astype(str).str.strip()
        tmp = tmp[tmp["no_collab_norm"] != ""].copy()
        parts.append(tmp)

    long_map = pd.concat(parts, ignore_index=True)
    long_map = long_map.drop_duplicates(subset=["branch", "no_collab_norm", "collab_master_id"]).copy()
    agg = (
        long_map.groupby(["branch", "no_collab_norm"], as_index=False)
        .agg(
            n_master=("collab_master_id", "nunique"),
            collab_master_id=("collab_master_id", "first"),
            collab_display_name=("collab_display_name", "first"),
        )
    )
    return agg


def attach_collab_master(df: pd.DataFrame, matched: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["no_collab_norm"] = out["No collaborateur"].apply(_norm_id)

    oe_col = _find_oe_column(out.columns)
    out["_branch_hint"] = out[oe_col].apply(_branch_hint_from_oe) if oe_col is not None else ""

    match_long = _make_match_long(matched)
    out = out.merge(
        match_long,
        how="left",
        left_on=["_branch_hint", "no_collab_norm"],
        right_on=["branch", "no_collab_norm"],
    )

    out["collab_match_status"] = ""
    out.loc[(out["n_master"] == 1) & out["collab_master_id"].notna(), "collab_match_status"] = "OK_BRANCH"
    out.loc[(out["n_master"] > 1), "collab_match_status"] = "AMBIG_MATCH"

    fallback = (
        match_long.groupby("no_collab_norm", as_index=False)
        .agg(
            n_master=("collab_master_id", "nunique"),
            collab_master_id=("collab_master_id", "first"),
            collab_display_name=("collab_display_name", "first"),
        )
    )
    fallback_unique = fallback[fallback["n_master"] == 1].copy()
    fallback_ambig  = fallback[fallback["n_master"] > 1].copy()

    unique_master = dict(zip(fallback_unique["no_collab_norm"], fallback_unique["collab_master_id"]))
    unique_name   = dict(zip(fallback_unique["no_collab_norm"], fallback_unique["collab_display_name"]))
    ambig_codes   = set(fallback_ambig["no_collab_norm"].astype(str).tolist())

    missing_branch_match = out["collab_master_id"].isna() & (out["no_collab_norm"].astype(str) != "")
    out.loc[missing_branch_match, "collab_master_id"] = out.loc[missing_branch_match, "no_collab_norm"].map(unique_master)
    out.loc[missing_branch_match, "collab_display_name"] = out.loc[missing_branch_match, "no_collab_norm"].map(unique_name)
    out.loc[missing_branch_match & out["collab_master_id"].notna(), "collab_match_status"] = "OK_FALLBACK"

    out.loc[
        (out["collab_match_status"] == "") &
        (out["no_collab_norm"].astype(str).isin(unique_master.keys())),
        "collab_match_status"
    ] = "UNIQUE_CODE_NO_BRANCH"

    out.loc[
        (out["collab_match_status"] == "") &
        (out["no_collab_norm"].astype(str).isin(list(ambig_codes))),
        "collab_match_status"
    ] = "AMBIG_MATCH"

    out.loc[out["no_collab_norm"].astype(str) == "", "collab_match_status"] = "MISSING_CODE"

    out["collab_display_name"] = out["collab_display_name"].where(
        out["collab_display_name"].astype(str).str.strip().ne(""),
        out["Collaborateur"].astype(str),
    )

    out["collab_master_id"] = out["collab_master_id"].astype(str).map(_clean_str)
    out["collab_key"] = out["collab_master_id"].astype(str).map(_clean_str)

    empty_key = out["collab_key"] == ""
    out.loc[empty_key & out["no_collab_norm"].astype(str).ne(""), "collab_key"] = (
        "UNMATCHED_CODE::" + out.loc[empty_key & out["no_collab_norm"].astype(str).ne(""), "no_collab_norm"].astype(str)
    )
    still_empty = out["collab_key"] == ""
    out.loc[still_empty, "collab_key"] = (
        "UNMATCHED_NAME::" + out.loc[still_empty, "collab_display_name"].map(_clean_name_key)
    )

    base_cols = list(df.columns)
    keep_cols = base_cols + [
        "no_collab_norm", "_branch_hint", "branch", "n_master",
        "collab_master_id", "collab_display_name", "collab_match_status", "collab_key"
    ]
    out = out[keep_cols].drop_duplicates().copy()
    return ensure_unique_columns(out)

In [5]:
#@title 5) Interval engine

def _to_local_naive_ts(value):
    if value is None:
        return pd.NaT
    ts = pd.to_datetime(value, errors="coerce")
    if pd.isna(ts):
        return pd.NaT
    ts = pd.Timestamp(ts)
    if ts.tzinfo is not None:
        return ts.tz_convert(SWISS_TZ).tz_localize(None)
    return ts


def _coerce_time_obj(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, pd.Timestamp):
        return value.time()
    if isinstance(value, datetime):
        return value.time()
    if isinstance(value, dtime):
        return value
    if hasattr(value, "hour") and hasattr(value, "minute") and hasattr(value, "second"):
        return dtime(value.hour, value.minute, value.second)
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return None
        for fmt in ("%H:%M:%S", "%H:%M"):
            try:
                return datetime.strptime(s, fmt).time()
            except ValueError:
                pass
    return None


def _combine_day_and_time_local(day_value, time_value):
    day_ts = pd.to_datetime(day_value, errors="coerce")
    if pd.isna(day_ts):
        return pd.NaT
    day_ts = pd.Timestamp(day_ts)
    if day_ts.tzinfo is not None:
        day_date = day_ts.tz_convert(SWISS_TZ).tz_localize(None).date()
    else:
        day_date = day_ts.date()
    tt = _coerce_time_obj(time_value)
    if tt is None:
        return pd.NaT
    return pd.Timestamp(datetime(day_date.year, day_date.month, day_date.day, tt.hour, tt.minute, tt.second))


def _build_interval_from_row(row):
    raw_start = _to_local_naive_ts(row.get("Début"))
    raw_end   = _to_local_naive_ts(row.get("Fin"))

    fb_start = _combine_day_and_time_local(row.get("Date début"), row.get("Heure début"))
    fb_end   = _combine_day_and_time_local(row.get("Date début"), row.get("Heure fin"))

    start_dt = raw_start if pd.notna(raw_start) else fb_start
    end_dt   = raw_end   if pd.notna(raw_end)   else fb_end
    source = "RAW_DATETIME" if pd.notna(raw_start) and pd.notna(raw_end) else "DATE_PLUS_TIME"

    if pd.isna(start_dt) or pd.isna(end_dt):
        return pd.Series({
            "start_dt_local": pd.NaT,
            "end_dt_local": pd.NaT,
            "row_interval_minutes": np.nan,
            "interval_status": "INVALID_MISSING_START_OR_END",
            "interval_source": source,
        })

    corrected_overnight = False
    if end_dt < start_dt:
        end_dt = end_dt + pd.Timedelta(days=1)
        corrected_overnight = True

    minutes = (end_dt - start_dt).total_seconds() / 60.0
    if minutes <= 0:
        status = "INVALID_NON_POSITIVE"
    elif minutes > float(MAX_SINGLE_INTERVAL_HOURS) * 60.0:
        status = f"INVALID_GT_{int(MAX_SINGLE_INTERVAL_HOURS)}H"
    else:
        status = "OK_OVERNIGHT" if corrected_overnight else "OK"

    return pd.Series({
        "start_dt_local": start_dt,
        "end_dt_local": end_dt,
        "row_interval_minutes": float(minutes) if np.isfinite(minutes) else np.nan,
        "interval_status": status,
        "interval_source": source,
    })


def add_interval_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    intervals = df.apply(_build_interval_from_row, axis=1)
    df = pd.concat([df, intervals], axis=1)
    df["interval_valid"] = df["interval_status"].isin(VALID_INTERVAL_STATUSES)

    no_prest = df["No prestation"].astype(str).str.strip()
    df["is_pause"] = no_prest.isin(PAUSE_CODES)
    df["is_transport"] = no_prest.isin(TRANSPORT_CODES)
    df["is_195"] = no_prest.isin(PSEUDO_NON_WORK_CODES)

    df["is_actual_work"] = (~df["is_pause"]) & (~df["is_transport"]) & (~df["is_195"])
    df["is_work_for_50h"] = df["is_actual_work"] | (df["is_transport"] if COUNT_TRANSPORT_AS_WORK_FOR_50H else False)
    df["is_work_for_streak"] = df["is_actual_work"] | (df["is_transport"] if COUNT_TRANSPORT_AS_WORK_FOR_STREAK else False)
    df["is_work_for_breaks"] = df["is_actual_work"] | (df["is_transport"] if COUNT_TRANSPORT_AS_WORK_FOR_BREAKS else False)
    df["is_work_for_service"] = df["is_actual_work"] | (df["is_transport"] if COUNT_TRANSPORT_FOR_SERVICE_BOUNDARY else False)

    df["row_start_date"] = pd.to_datetime(df["start_dt_local"], errors="coerce").dt.date
    df["row_end_date"] = pd.to_datetime(df["end_dt_local"], errors="coerce").dt.date
    return df

In [6]:
#@title 6) Interval math helpers

def _merge_intervals(intervals):
    clean = []
    for s, e in intervals or []:
        if pd.isna(s) or pd.isna(e):
            continue
        s = pd.Timestamp(s)
        e = pd.Timestamp(e)
        if e > s:
            clean.append((s, e))
    if not clean:
        return []
    clean = sorted(clean, key=lambda x: x[0])
    merged = [clean[0]]
    for s, e in clean[1:]:
        ls, le = merged[-1]
        if s <= le:
            merged[-1] = (ls, max(le, e))
        else:
            merged.append((s, e))
    return merged


def _interval_minutes(intervals):
    return float(sum((e - s).total_seconds() / 60.0 for s, e in (intervals or [])))


def _gap_minutes(intervals):
    if not intervals or len(intervals) < 2:
        return 0.0
    total = 0.0
    for i in range(len(intervals) - 1):
        gap = (intervals[i + 1][0] - intervals[i][1]).total_seconds() / 60.0
        if gap > 0:
            total += gap
    return float(total)


def _longest_internal_interruption_minutes(work_intervals):
    if not work_intervals or len(work_intervals) < 2:
        return 0.0
    longest = 0.0
    for i in range(len(work_intervals) - 1):
        gap = (work_intervals[i + 1][0] - work_intervals[i][1]).total_seconds() / 60.0
        if gap > longest:
            longest = gap
    return float(longest)


def _subtract_intervals(base_intervals, subtract_intervals):
    base = _merge_intervals(base_intervals)
    subs = _merge_intervals(subtract_intervals)
    if not base:
        return []
    if not subs:
        return base

    result = []
    for b_s, b_e in base:
        segments = [(b_s, b_e)]
        for p_s, p_e in subs:
            new_segments = []
            for s, e in segments:
                if p_e <= s or p_s >= e:
                    new_segments.append((s, e))
                    continue
                if p_s > s:
                    new_segments.append((s, min(p_s, e)))
                if p_e < e:
                    new_segments.append((max(p_e, s), e))
            segments = [(s, e) for s, e in new_segments if e > s]
            if not segments:
                break
        result.extend(segments)
    return _merge_intervals(result)


def _clip_intervals_to_window(intervals, start, end):
    out = []
    for s, e in intervals or []:
        cs = max(pd.Timestamp(s), pd.Timestamp(start))
        ce = min(pd.Timestamp(e), pd.Timestamp(end))
        if ce > cs:
            out.append((cs, ce))
    return out


def _intervals_from_df(df, flag_col):
    if df is None or df.empty:
        return []
    mask = df[flag_col].fillna(False) & df["interval_valid"].fillna(False)
    return _merge_intervals([
        (s, e)
        for s, e in df.loc[mask, ["start_dt_local", "end_dt_local"]].itertuples(index=False, name=None)
        if pd.notna(s) and pd.notna(e)
    ])


def _pause_intervals_from_df(df):
    if df is None or df.empty:
        return []
    mask = df["is_pause"].fillna(False) & df["interval_valid"].fillna(False)
    return _merge_intervals([
        (s, e)
        for s, e in df.loc[mask, ["start_dt_local", "end_dt_local"]].itertuples(index=False, name=None)
        if pd.notna(s) and pd.notna(e)
    ])


def _intervals_to_text(intervals, limit=8):
    vals = []
    for s, e in (intervals or [])[:limit]:
        vals.append(f"{pd.Timestamp(s).strftime('%d.%m.%Y %H:%M')} → {pd.Timestamp(e).strftime('%d.%m.%Y %H:%M')}")
    extra = "" if len(intervals or []) <= limit else f" … +{len(intervals)-limit} more"
    return " | ".join(vals) + extra


def _week_monday(date_value):
    d = pd.Timestamp(date_value).normalize()
    return (d - pd.Timedelta(days=int(d.weekday()))).date()


def _month_str(date_value):
    if pd.isna(date_value):
        return ""
    return pd.Timestamp(date_value).strftime("%Y-%m")

In [7]:
#@title 7) Hybrid service builder — FAST VERSION

# This replaces the first hybrid builder.
# Main speed fix: when attaching rows to services, it only scans the current employee's rows,
# not the entire 127k-row input table for every service.

def _pick_mode_or_first(s):
    ss = s.dropna().astype(str).str.strip()
    ss = ss[~ss.str.lower().isin(["", "nan", "none", "nat", "<na>"])]
    if ss.empty:
        return ""
    mode = ss.mode()
    return str(mode.iloc[0]) if not mode.empty else str(ss.iloc[0])


def _clean_codes_list(s):
    ss = s.dropna().astype(str).str.strip()
    ss = ss[~ss.str.lower().isin(["", "nan", "none", "nat", "<na>"])]
    return ", ".join(sorted(set(ss.tolist())))


def _employee_services_from_work_intervals(g: pd.DataFrame):
    reset_gap = pd.Timedelta(hours=float(SERVICE_RESET_GAP_HOURS))
    work_rows = g[g["interval_valid"].fillna(False) & g["is_work_for_service"].fillna(False)].copy()
    work_rows = work_rows.sort_values(["start_dt_local", "end_dt_local"])

    intervals = []
    for idx, r in work_rows[["start_dt_local", "end_dt_local"]].iterrows():
        s = pd.to_datetime(r["start_dt_local"], errors="coerce")
        e = pd.to_datetime(r["end_dt_local"], errors="coerce")
        if pd.notna(s) and pd.notna(e) and e > s:
            intervals.append((idx, pd.Timestamp(s), pd.Timestamp(e)))

    services = []
    current = None
    seq = 0

    for idx, s, e in intervals:
        if current is None:
            seq += 1
            current = {"seq": seq, "start": s, "end": e, "work_row_indices": [idx]}
            continue

        gap = s - current["end"]
        if gap >= reset_gap:
            services.append(current)
            seq += 1
            current = {"seq": seq, "start": s, "end": e, "work_row_indices": [idx]}
        else:
            current["end"] = max(current["end"], e)
            current["work_row_indices"].append(idx)

    if current is not None:
        services.append(current)

    return services


def build_services_and_tagged_rows(df: pd.DataFrame):
    df = df.copy()

    for c in [
        "service_id", "service_date", "service_start", "service_end",
        "continuation_from_previous_service", "creates_worked_day"
    ]:
        df[c] = pd.NA

    service_rows = []
    orphan_pause_rows = []

    total_groups = df["collab_key"].nunique(dropna=False)
    processed_groups = 0

    for ckey, g in df.groupby("collab_key", dropna=False, sort=False):
        processed_groups += 1
        if processed_groups % 50 == 0:
            print(f"  ...service groups processed: {processed_groups}/{total_groups}")

        g = g.sort_values(["start_dt_local", "end_dt_local"]).copy()
        services = _employee_services_from_work_intervals(g)

        valid_g = g[g["interval_valid"].fillna(False)].copy()
        if valid_g.empty:
            continue

        valid_starts = pd.to_datetime(valid_g["start_dt_local"], errors="coerce")
        valid_ends = pd.to_datetime(valid_g["end_dt_local"], errors="coerce")

        for svc in services:
            service_start = pd.Timestamp(svc["start"])
            service_end = pd.Timestamp(svc["end"])
            service_date = service_start.date()
            service_id = f"{str(ckey)}::{service_start.strftime('%Y%m%d_%H%M')}::{svc['seq']:03d}"

            # Attach any valid row from THIS SAME COLLABORATOR whose interval overlaps the service envelope.
            overlap_mask = (valid_starts < service_end) & (valid_ends > service_start)
            attach_idx = valid_g.index[overlap_mask].tolist()
            if not attach_idx:
                continue

            # Services are normally non-overlapping, but this protects unusual data overlaps.
            for idx in attach_idx:
                existing = _clean_str(df.at[idx, "service_id"])
                if existing:
                    old_s = pd.Timestamp(df.at[idx, "service_start"])
                    old_e = pd.Timestamp(df.at[idx, "service_end"])
                    row_s = pd.Timestamp(df.at[idx, "start_dt_local"])
                    row_e = pd.Timestamp(df.at[idx, "end_dt_local"])
                    old_overlap = max(0.0, (min(old_e, row_e) - max(old_s, row_s)).total_seconds())
                    new_overlap = max(0.0, (min(service_end, row_e) - max(service_start, row_s)).total_seconds())
                    if new_overlap <= old_overlap:
                        continue

                df.at[idx, "service_id"] = service_id
                df.at[idx, "service_date"] = service_date
                df.at[idx, "service_start"] = service_start
                df.at[idx, "service_end"] = service_end
                row_start_date = pd.Timestamp(df.at[idx, "start_dt_local"]).date()
                df.at[idx, "continuation_from_previous_service"] = bool(row_start_date > service_date)

            svc_df = df.loc[attach_idx].copy()
            svc_df = svc_df[svc_df["service_id"].astype(str) == service_id].copy()
            if svc_df.empty:
                continue

            pause_intervals = _pause_intervals_from_df(svc_df)
            work_50_raw = _intervals_from_df(svc_df, "is_work_for_50h")
            work_streak_raw = _intervals_from_df(svc_df, "is_work_for_streak")
            work_breaks_raw = _intervals_from_df(svc_df, "is_work_for_breaks")
            actual_work_raw = _intervals_from_df(svc_df, "is_actual_work")

            net_50 = _subtract_intervals(work_50_raw, pause_intervals)
            net_streak = _subtract_intervals(work_streak_raw, pause_intervals)
            net_breaks = _subtract_intervals(work_breaks_raw, pause_intervals)
            net_actual = _subtract_intervals(actual_work_raw, pause_intervals)

            creates_worked_day = _interval_minutes(net_streak) > 0
            df.loc[svc_df.index, "creates_worked_day"] = bool(creates_worked_day)

            collab_name = _pick_mode_or_first(svc_df["collab_display_name"])
            collab_id = _pick_mode_or_first(svc_df["collab_master_id"])
            codes_str = _clean_codes_list(svc_df["no_collab_norm"])
            worst_status = pick_worst_status(svc_df["collab_match_status"].astype(str).tolist())

            attached_start_dates = sorted(set(pd.to_datetime(svc_df["start_dt_local"], errors="coerce").dt.date.dropna().tolist()))
            continuation_row_count = int((pd.to_datetime(svc_df["start_dt_local"], errors="coerce").dt.date > service_date).sum())

            service_rows.append({
                "service_id": service_id,
                "collab_uid": ckey,
                "collab_key": ckey,
                "Collaborateur": collab_name,
                "Collaborateur_ID": collab_id,
                "No_collaborateur_codes": codes_str,
                "Match_status": worst_status,
                "service_date": service_date,
                "service_month": _month_str(service_date),
                "service_start": service_start,
                "service_end": service_end,
                "amplitude_minutes": (service_end - service_start).total_seconds() / 60.0,
                "amplitude_hours": (service_end - service_start).total_seconds() / 3600.0,
                "continues_after_midnight": bool(service_end.date() > service_start.date()),
                "attached_calendar_dates": ", ".join(str(x) for x in attached_start_dates),
                "continuation_row_count": continuation_row_count,
                "creates_worked_day": bool(creates_worked_day),
                "net_50h_minutes": _interval_minutes(net_50),
                "net_streak_minutes": _interval_minutes(net_streak),
                "net_breaks_minutes": _interval_minutes(net_breaks),
                "net_actual_work_minutes": _interval_minutes(net_actual),
                "pause_minutes_inside_service": _interval_minutes(pause_intervals),
                "internal_gaps_minutes_net_breaks": _gap_minutes(net_breaks),
                "longest_internal_interruption_minutes": _longest_internal_interruption_minutes(net_breaks),
                "work_50h_intervals_text": _intervals_to_text(net_50),
                "work_breaks_intervals_text": _intervals_to_text(net_breaks),
                "pause_intervals_text": _intervals_to_text(pause_intervals),
                "raw_rows_attached": int(len(svc_df)),
                "raw_pause_rows_attached": int(svc_df["is_pause"].sum()),
                "raw_work_rows_attached": int(svc_df["is_work_for_service"].sum()),
                "_net_50_intervals": net_50,
                "_net_breaks_intervals": net_breaks,
            })

        orphan = valid_g[
            valid_g["is_pause"].fillna(False) &
            df.loc[valid_g.index, "service_id"].astype(str).isin(["", "<NA>", "nan", "None"])
        ].copy()
        if not orphan.empty:
            orphan_pause_rows.append(orphan.copy())

    services_df = pd.DataFrame(service_rows)
    if not services_df.empty:
        services_df = services_df.sort_values(["Collaborateur", "service_start"]).reset_index(drop=True)

    orphan_pauses_df = pd.concat(orphan_pause_rows, ignore_index=False).copy() if orphan_pause_rows else pd.DataFrame()

    print("✅ Services built:", len(services_df))
    print("✅ Orphan pause rows:", len(orphan_pauses_df))

    return services_df, df, orphan_pauses_df

In [8]:
#@title 8) Calendar slices for weekly/monthly hour allocation

def split_interval_at_midnights(s, e):
    s = pd.Timestamp(s)
    e = pd.Timestamp(e)
    out = []
    cur = s
    while cur < e:
        next_midnight = pd.Timestamp(cur.date()) + pd.Timedelta(days=1)
        seg_end = min(e, next_midnight)
        if seg_end > cur:
            out.append((cur, seg_end))
        cur = seg_end
    return out


def build_calendar_slices(services_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if services_df is None or services_df.empty:
        return pd.DataFrame()

    for _, svc in services_df.iterrows():
        intervals = svc.get("_net_50_intervals", [])
        for s, e in intervals:
            for ss, ee in split_interval_at_midnights(s, e):
                cal_date = ss.date()
                service_date = svc["service_date"]
                rows.append({
                    "service_id": svc["service_id"],
                    "collab_uid": svc["collab_uid"],
                    "collab_key": svc["collab_key"],
                    "Collaborateur": svc["Collaborateur"],
                    "No_collaborateur_codes": svc["No_collaborateur_codes"],
                    "Match_status": svc["Match_status"],
                    "service_date": service_date,
                    "calendar_date": cal_date,
                    "target_month": _month_str(cal_date),
                    "week_monday": _week_monday(cal_date),
                    "slice_start": ss,
                    "slice_end": ee,
                    "minutes": (ee - ss).total_seconds() / 60.0,
                    "hours": (ee - ss).total_seconds() / 3600.0,
                    "continuation_from_previous_service": bool(cal_date > service_date),
                    "note": "overnight continuation allocated to actual calendar date" if cal_date > service_date else "service start day",
                })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["Collaborateur", "slice_start"]).reset_index(drop=True)
    return out

In [9]:
#@title 9) LTr checks using hybrid service/calendar model

def _common_infraction_row(rule, event_date, service_or_row, severity, detail, target_month=None):
    event_date_val = pd.Timestamp(event_date).date() if pd.notna(event_date) else None
    return {
        "TARGET_MONTH": target_month or (_month_str(event_date_val) if event_date_val else ""),
        "EVENT_DATE": event_date_val,
        "RULE": rule,
        "SEVERITY": severity,
        "DETAIL": detail,
        "collab_uid": service_or_row.get("collab_uid", service_or_row.get("collab_key", "")),
        "Collaborateur": service_or_row.get("Collaborateur", ""),
        "No_collaborateur_codes": service_or_row.get("No_collaborateur_codes", ""),
        "Match_status": service_or_row.get("Match_status", ""),
        "service_id": service_or_row.get("service_id", ""),
        "service_start": service_or_row.get("service_start", pd.NaT),
        "service_end": service_or_row.get("service_end", pd.NaT),
        "service_date": service_or_row.get("service_date", pd.NaT),
    }


def check_over_50h(calendar_slices: pd.DataFrame):
    if not RUN_OVER50H or calendar_slices is None or calendar_slices.empty:
        return pd.DataFrame(), pd.DataFrame()

    weekly = (
        calendar_slices.groupby(["collab_key", "week_monday"], dropna=False)
        .agg(
            collab_uid=("collab_uid", "first"),
            Collaborateur=("Collaborateur", "first"),
            No_collaborateur_codes=("No_collaborateur_codes", "first"),
            Match_status=("Match_status", "first"),
            week_work_minutes=("minutes", "sum"),
            first_slice_start=("slice_start", "min"),
            last_slice_end=("slice_end", "max"),
            services_in_week=("service_id", lambda x: len(set(x.astype(str))))
        )
        .reset_index()
    )
    weekly["week_sunday"] = pd.to_datetime(weekly["week_monday"]) + pd.to_timedelta(6, unit="D")
    weekly["week_work_hours"] = weekly["week_work_minutes"] / 60.0
    weekly["excess_hours"] = weekly["week_work_hours"] - float(WEEKLY_LIMIT_HOURS)
    weekly["TARGET_MONTH"] = pd.to_datetime(weekly["week_monday"]).dt.strftime("%Y-%m")

    infra = weekly[weekly["week_work_hours"] > float(WEEKLY_LIMIT_HOURS)].copy()
    rows = []
    for _, r in infra.iterrows():
        rows.append(_common_infraction_row(
            rule=f"OVER_{int(WEEKLY_LIMIT_HOURS)}H_WEEK",
            event_date=r["week_monday"],
            service_or_row=r,
            severity="Potential infraction",
            detail=f"Week {r['week_monday']} → {pd.Timestamp(r['week_sunday']).date()}: {r['week_work_hours']:.2f}h > {WEEKLY_LIMIT_HOURS:.0f}h by {r['excess_hours']:.2f}h. Hours allocated by actual calendar date slices.",
            target_month=r["TARGET_MONTH"],
        ))
    all_rows = pd.DataFrame(rows)
    return infra, all_rows


def check_streak_7days(services_df: pd.DataFrame):
    if not RUN_STREAK or services_df is None or services_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    rows = []
    for ckey, g in services_df[services_df["creates_worked_day"] == True].groupby("collab_key", dropna=False):
        g = g.sort_values("service_start")
        days = sorted(set(pd.to_datetime(g["service_date"], errors="coerce").dropna().dt.date.tolist()))
        if not days:
            continue

        streak = [days[0]]
        streaks = []
        for d in days[1:]:
            if (d - streak[-1]).days == 1:
                streak.append(d)
            else:
                if len(streak) >= 7:
                    streaks.append(streak.copy())
                streak = [d]
        if len(streak) >= 7:
            streaks.append(streak.copy())

        for st in streaks:
            first_svc = g[pd.to_datetime(g["service_date"]).dt.date.isin(st)].iloc[0]
            rows.append({
                "TARGET_MONTH": _month_str(st[0]),
                "EVENT_DATE": st[0],
                "RULE": "STREAK_7DAYS",
                "SEVERITY": "Potential infraction",
                "DETAIL": f"{len(st)} consecutive service-start days: {st[0]} → {st[-1]}. Overnight continuation dates do not create extra worked days.",
                "collab_uid": first_svc["collab_uid"],
                "collab_key": ckey,
                "Collaborateur": first_svc["Collaborateur"],
                "No_collaborateur_codes": first_svc["No_collaborateur_codes"],
                "Match_status": first_svc["Match_status"],
                "Jour de début": st[0],
                "Jour de fin": st[-1],
                "Nombre de jours consécutifs": len(st),
            })

    infra = pd.DataFrame(rows)
    if infra.empty:
        return pd.DataFrame(), pd.DataFrame()
    all_rows = infra.copy()
    return infra, all_rows


def check_span_over_14h(services_df: pd.DataFrame):
    if not RUN_SPAN or services_df is None or services_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    base = services_df[pd.to_numeric(services_df["amplitude_hours"], errors="coerce") > float(SPAN_LIMIT_HOURS)].copy()
    if base.empty:
        return pd.DataFrame(), pd.DataFrame()

    base["TARGET_MONTH"] = pd.to_datetime(base["service_date"]).dt.strftime("%Y-%m")
    base["EVENT_DATE"] = base["service_date"]
    base["RULE"] = f"SPAN_OVER_{int(SPAN_LIMIT_HOURS)}H"
    base["SEVERITY"] = "Potential infraction"
    base["DETAIL"] = base.apply(
        lambda r: f"Service amplitude {r['amplitude_hours']:.2f}h from {r['service_start']} to {r['service_end']}. Early-morning continuation remains attached to service date {r['service_date']}.",
        axis=1
    )
    all_rows = base[[
        "TARGET_MONTH", "EVENT_DATE", "RULE", "SEVERITY", "DETAIL",
        "collab_uid", "collab_key", "Collaborateur", "No_collaborateur_codes", "Match_status",
        "service_id", "service_date", "service_start", "service_end", "amplitude_hours",
        "continues_after_midnight", "continuation_row_count", "attached_calendar_dates"
    ]].copy()
    return base, all_rows


def check_rest_under_11h(services_df: pd.DataFrame):
    if not RUN_REST11 or services_df is None or services_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    rest = services_df.sort_values(["collab_key", "service_start"]).copy()
    rest["previous_service_end"] = rest.groupby("collab_key")["service_end"].shift(1)
    rest["previous_service_date"] = rest.groupby("collab_key")["service_date"].shift(1)
    rest["rest_pre_hours"] = (pd.to_datetime(rest["service_start"]) - pd.to_datetime(rest["previous_service_end"])).dt.total_seconds() / 3600.0
    rest["week_monday"] = pd.to_datetime(rest["service_date"]).apply(lambda x: _week_monday(x) if pd.notna(x) else None)
    rest["week_sunday"] = pd.to_datetime(rest["week_monday"]) + pd.to_timedelta(6, unit="D")

    rest["rest_pre_allow"] = np.where(rest["rest_pre_hours"].notna(), np.minimum(rest["rest_pre_hours"], REST_NORMAL_HOURS), np.nan)
    rest["roll14_back_true_hours"] = np.nan
    rest["roll14_back_days_present"] = 0

    for ckey, g in rest.groupby("collab_key", sort=False):
        g = g.sort_values("service_start")
        for ix, row in g.iterrows():
            cur_date = pd.Timestamp(row["service_date"])
            start_date = cur_date - pd.Timedelta(days=14)
            end_date = cur_date - pd.Timedelta(days=1)
            mask = (pd.to_datetime(g["service_date"]) >= start_date) & (pd.to_datetime(g["service_date"]) <= end_date)
            vals = g.loc[mask, "rest_pre_hours"].dropna()
            rest.at[ix, "roll14_back_days_present"] = int(vals.shape[0])
            if len(vals):
                rest.at[ix, "roll14_back_true_hours"] = float(vals.mean())

    rest["lt8"] = rest["rest_pre_hours"] < REST_ABSOLUTE_MIN_HOURS
    rest["lt11"] = rest["rest_pre_hours"] < REST_NORMAL_HOURS
    rest["btw8_11"] = (
        (rest["rest_pre_hours"] >= REST_ABSOLUTE_MIN_HOURS) &
        (rest["rest_pre_hours"] < REST_NORMAL_HOURS)
    )

    rest = rest.sort_values(["collab_key", "week_monday", "service_start"]).copy()
    rest["_btw8_11_int"] = rest["btw8_11"].fillna(False).astype(int)
    rest["idx_8to11_in_week"] = rest.groupby(["collab_key", "week_monday"], dropna=False)["_btw8_11_int"].cumsum()
    rest.loc[~rest["btw8_11"].fillna(False), "idx_8to11_in_week"] = np.nan
    rest.drop(columns=["_btw8_11_int"], inplace=True)

    rest["rest_decision"] = "OK"
    rest["rest_reason"] = ""

    mask_missing_prev = rest["previous_service_end"].isna()
    rest.loc[mask_missing_prev, "rest_decision"] = "NO_PREVIOUS_SERVICE"
    rest.loc[mask_missing_prev, "rest_reason"] = "No previous service in file, so pre-service rest cannot be checked."

    mask_same_day = rest["service_date"] == rest["previous_service_date"]
    rest.loc[mask_same_day, "rest_decision"] = "SAME_DAY_SERVICES"
    rest.loc[mask_same_day, "rest_reason"] = "Services fall on the same day (split shift); daily rest rule applies between distinct work days."

    mask_negative = (rest["rest_pre_hours"] < 0) & (~mask_same_day)
    rest.loc[mask_negative, "rest_decision"] = "INFRACTION_OR_DATA_ERROR"
    rest.loc[mask_negative, "rest_reason"] = "Service overlaps previous service or negative rest; check source data."

    mask_lt8 = rest["lt8"].fillna(False) & (~mask_negative) & (~mask_same_day)
    rest.loc[mask_lt8, "rest_decision"] = "INFRACTION"
    rest.loc[mask_lt8, "rest_reason"] = "Rest is strictly below 8h."

    mask_8to11 = rest["btw8_11"].fillna(False) & (~mask_same_day)
    first_in_week = rest["idx_8to11_in_week"] == 1
    avg_known = rest["roll14_back_true_hours"].notna()
    avg_ok = rest["roll14_back_true_hours"] >= REST_NORMAL_HOURS

    allowed_mask = mask_8to11 & first_in_week & avg_known & avg_ok
    rest.loc[allowed_mask, "rest_decision"] = "ALLOWED_REDUCED_REST"
    rest.loc[allowed_mask, "rest_reason"] = "First 8–11h reduced rest in week and backward 14-day average is >= 11h."

    missing_avg_mask = mask_8to11 & first_in_week & (~avg_known) & REQUIRE_AVG_FOR_REDUCED_REST
    rest.loc[missing_avg_mask, "rest_decision"] = "REVIEW_MISSING_14D_CONTEXT"
    rest.loc[missing_avg_mask, "rest_reason"] = "8–11h reduced rest, first in week, but 14-day average cannot be proven from available history."

    avg_bad_mask = mask_8to11 & first_in_week & avg_known & (~avg_ok)
    rest.loc[avg_bad_mask, "rest_decision"] = "INFRACTION"
    rest.loc[avg_bad_mask, "rest_reason"] = "8–11h reduced rest but backward 14-day average is < 11h."

    second_reduction_mask = mask_8to11 & (~first_in_week)
    rest.loc[second_reduction_mask, "rest_decision"] = "INFRACTION"
    rest.loc[second_reduction_mask, "rest_reason"] = "Second or later 8–11h reduced rest in the same Monday–Sunday week."

    infra = rest[rest["rest_decision"].isin(["INFRACTION", "INFRACTION_OR_DATA_ERROR"])].copy()
    review = rest[rest["rest_decision"].isin(["ALLOWED_REDUCED_REST", "REVIEW_MISSING_14D_CONTEXT"])].copy()

    for d in [infra, review]:
        if not d.empty:
            d["TARGET_MONTH"] = pd.to_datetime(d["service_date"]).dt.strftime("%Y-%m")
            d["EVENT_DATE"] = d["service_date"]
            d["RULE"] = "REST_UNDER_11H"
            d["SEVERITY"] = d["rest_decision"]
            d["DETAIL"] = d.apply(
                lambda r: f"Rest before service: {r['rest_pre_hours']:.2f}h. Decision: {r['rest_decision']}. {r['rest_reason']}",
                axis=1,
            )

    all_rows = infra[[
        "TARGET_MONTH", "EVENT_DATE", "RULE", "SEVERITY", "DETAIL",
        "collab_uid", "collab_key", "Collaborateur", "No_collaborateur_codes", "Match_status",
        "service_id", "service_date", "previous_service_end", "service_start", "service_end",
        "rest_pre_hours", "week_monday", "week_sunday", "roll14_back_true_hours",
        "roll14_back_days_present", "idx_8to11_in_week", "rest_decision", "rest_reason"
    ]].copy() if not infra.empty else pd.DataFrame()

    review_rows = review[[
        "TARGET_MONTH", "EVENT_DATE", "RULE", "SEVERITY", "DETAIL",
        "collab_uid", "collab_key", "Collaborateur", "No_collaborateur_codes", "Match_status",
        "service_id", "service_date", "previous_service_end", "service_start", "service_end",
        "rest_pre_hours", "week_monday", "week_sunday", "roll14_back_true_hours",
        "roll14_back_days_present", "idx_8to11_in_week", "rest_decision", "rest_reason"
    ]].copy() if not review.empty else pd.DataFrame()

    return infra, all_rows, review_rows


def required_interruption_minutes(work_minutes: float) -> int:
    # Strict thresholds: more than 5h30, more than 7h, more than 9h.
    if work_minutes >= 9 * 60:
        return 60
    if work_minutes >= 7 * 60:
        return 30
    if work_minutes >= 5 * 60 + 30:
        return 15
    return 0


def check_breaks(services_df: pd.DataFrame):
    if not RUN_BREAKS or services_df is None or services_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    rows = []
    audit_rows = []
    for _, svc in services_df.iterrows():
        net_breaks = svc.get("_net_breaks_intervals", [])
        work_min = _interval_minutes(net_breaks)
        required_min = required_interruption_minutes(work_min)

        total_interruption = _gap_minutes(net_breaks)
        longest_gap = _longest_internal_interruption_minutes(net_breaks)
        infraction = required_min > 0 and (total_interruption + 1e-9) < required_min

        audit = {
            "TARGET_MONTH": _month_str(svc["service_date"]),
            "EVENT_DATE": svc["service_date"],
            "collab_uid": svc["collab_uid"],
            "collab_key": svc["collab_key"],
            "Collaborateur": svc["Collaborateur"],
            "No_collaborateur_codes": svc["No_collaborateur_codes"],
            "Match_status": svc["Match_status"],
            "service_id": svc["service_id"],
            "service_date": svc["service_date"],
            "service_start": svc["service_start"],
            "service_end": svc["service_end"],
            "continues_after_midnight": svc["continues_after_midnight"],
            "work_minutes_for_break_rule": round(work_min, 1),
            "work_hours_for_break_rule": round(work_min / 60.0, 3),
            "required_interruption_minutes": int(required_min),
            "total_interruption_minutes": round(total_interruption, 1),
            "longest_internal_interruption_minutes": round(longest_gap, 1),
            "pause_minutes_inside_service": round(float(svc["pause_minutes_inside_service"]), 1),
            "work_intervals_after_pause_subtraction": svc["work_breaks_intervals_text"],
            "pause_intervals_inside_service": svc["pause_intervals_text"],
            "Infraction": "Pause insuffisante" if infraction else "",
        }
        audit_rows.append(audit)

        if infraction:
            rows.append({
                **audit,
                "RULE": "PAUSE_INSUFF",
                "SEVERITY": "Potential infraction",
                "DETAIL": f"Service work {work_min/60.0:.2f}h requires {required_min} min total interruption, but total internal interruption is only {total_interruption:.1f} min. Checked on full service, not calendar day.",
            })

    infra = pd.DataFrame(rows)
    audit = pd.DataFrame(audit_rows)
    all_rows = infra.copy()
    return infra, all_rows, audit


def build_data_quality(df_tagged: pd.DataFrame, orphan_pauses_df: pd.DataFrame):
    parts = []
    invalid = df_tagged[~df_tagged["interval_valid"].fillna(False)].copy()
    if not invalid.empty:
        invalid["QUALITY_TYPE"] = "INVALID_INTERVAL"
        invalid["DETAIL"] = invalid["interval_status"].astype(str)
        parts.append(invalid)

    if orphan_pauses_df is not None and not orphan_pauses_df.empty:
        op = orphan_pauses_df.copy()
        op["QUALITY_TYPE"] = "ORPHAN_PAUSE_NO_SERVICE"
        op["DETAIL"] = "Pause row is valid but does not overlap any work service; it does not create a worked day."
        parts.append(op)

    if not parts:
        return pd.DataFrame()

    q = pd.concat(parts, ignore_index=False, sort=False)
    q["EVENT_DATE"] = pd.to_datetime(q["start_dt_local"], errors="coerce").dt.date
    q["TARGET_MONTH"] = pd.to_datetime(q["start_dt_local"], errors="coerce").dt.strftime("%Y-%m")
    keep = [
        "TARGET_MONTH", "EVENT_DATE", "QUALITY_TYPE", "DETAIL",
        "Collaborateur", "No collaborateur", "No prestation", "Prestation",
        "Date début", "Heure début", "Heure fin", "Début", "Fin",
        "start_dt_local", "end_dt_local", "row_interval_minutes", "interval_status",
        "collab_key", "collab_display_name", "collab_match_status",
        "service_id", "service_date", "continuation_from_previous_service"
    ]
    return q[[c for c in keep if c in q.columns]].copy()

In [10]:

#@title 10) Load, compute, export one workbook

print("\n===== LOAD INPUTS =====")
df_raw = load_and_normalize(MERGED_RDA_PATH)
matched = load_matched_collabs(MATCHED_COLLABS_PATH)
df_raw = attach_collab_master(df_raw, matched)

df = add_interval_columns(df_raw)

print("✅ Loaded rows:", len(df))
print("Match status counts:")
print(df["collab_match_status"].value_counts(dropna=False).head(20))
print("\nInterval status counts:")
print(df["interval_status"].value_counts(dropna=False))

print("\n===== BUILD HYBRID SERVICES =====")
services_df, df_tagged, orphan_pauses_df = build_services_and_tagged_rows(df)
calendar_slices_df = build_calendar_slices(services_df)

print("✅ Services built:", len(services_df))
print("✅ Calendar hour slices:", len(calendar_slices_df))
print("✅ Orphan pauses not counted as workdays:", len(orphan_pauses_df) if orphan_pauses_df is not None else 0)

if not services_df.empty:
    print("Service months:", sorted(services_df["service_month"].dropna().astype(str).unique().tolist()))
if not calendar_slices_df.empty:
    print("Calendar months with hours:", sorted(calendar_slices_df["target_month"].dropna().astype(str).unique().tolist()))

print("\n===== RUN CHECKS =====")
over50_detail, over50_all = check_over_50h(calendar_slices_df)
streak_detail, streak_all = check_streak_7days(services_df)
span_detail, span_all = check_span_over_14h(services_df)
rest_detail, rest_all, rest_review = check_rest_under_11h(services_df)
breaks_detail, breaks_all, breaks_audit = check_breaks(services_df)
data_quality = build_data_quality(df_tagged, orphan_pauses_df)

all_infraction_parts = [x for x in [over50_all, streak_all, span_all, rest_all, breaks_all] if x is not None and not x.empty]
if all_infraction_parts:
    all_infractions = pd.concat(all_infraction_parts, ignore_index=True, sort=False)
    all_infractions = ensure_unique_columns(all_infractions)
    sort_cols = [c for c in ["TARGET_MONTH", "EVENT_DATE", "Collaborateur", "RULE"] if c in all_infractions.columns]
    if sort_cols:
        all_infractions = all_infractions.sort_values(sort_cols, kind="stable").reset_index(drop=True)
else:
    all_infractions = pd.DataFrame(columns=["TARGET_MONTH", "EVENT_DATE", "RULE", "SEVERITY", "DETAIL", "collab_uid", "Collaborateur"])

# Summary by month includes every month present in either raw input, service starts, calendar slices, or infractions.
raw_months = set(df["Mois"].dropna().astype(str).tolist()) if "Mois" in df.columns else set()
service_months = set(services_df["service_month"].dropna().astype(str).tolist()) if not services_df.empty else set()
calendar_months = set(calendar_slices_df["target_month"].dropna().astype(str).tolist()) if not calendar_slices_df.empty else set()
inf_months = set(all_infractions["TARGET_MONTH"].dropna().astype(str).tolist()) if not all_infractions.empty else set()
months = sorted(raw_months | service_months | calendar_months | inf_months)

summary_rows = []
for mm in months:
    row = {"TARGET_MONTH": mm}
    for rule in [f"OVER_{int(WEEKLY_LIMIT_HOURS)}H_WEEK", "STREAK_7DAYS", f"SPAN_OVER_{int(SPAN_LIMIT_HOURS)}H", "REST_UNDER_11H", "PAUSE_INSUFF"]:
        row[rule] = int(((all_infractions.get("TARGET_MONTH", pd.Series(dtype=str)).astype(str) == mm) & (all_infractions.get("RULE", pd.Series(dtype=str)).astype(str) == rule)).sum()) if not all_infractions.empty else 0
    row["TOTAL_INFRACTIONS"] = int((all_infractions["TARGET_MONTH"].astype(str) == mm).sum()) if not all_infractions.empty else 0
    row["SERVICES_STARTED"] = int((services_df["service_month"].astype(str) == mm).sum()) if not services_df.empty else 0
    row["CALENDAR_HOURS"] = float(calendar_slices_df.loc[calendar_slices_df["target_month"].astype(str) == mm, "hours"].sum()) if not calendar_slices_df.empty else 0.0
    row["ORPHAN_PAUSE_ROWS"] = int((data_quality.get("TARGET_MONTH", pd.Series(dtype=str)).astype(str).eq(mm) & data_quality.get("QUALITY_TYPE", pd.Series(dtype=str)).astype(str).eq("ORPHAN_PAUSE_NO_SERVICE")).sum()) if not data_quality.empty else 0
    row["INVALID_INTERVAL_ROWS"] = int((data_quality.get("TARGET_MONTH", pd.Series(dtype=str)).astype(str).eq(mm) & data_quality.get("QUALITY_TYPE", pd.Series(dtype=str)).astype(str).eq("INVALID_INTERVAL")).sum()) if not data_quality.empty else 0
    summary_rows.append(row)
summary_by_month = pd.DataFrame(summary_rows)

print("Over 50h weeks:", len(over50_detail))
print("7-day streaks:", len(streak_detail))
print("Span >14h services:", len(span_detail))
print("Rest infractions:", len(rest_detail))
print("Break infractions:", len(breaks_detail))
print("Total hard/potential infractions:", len(all_infractions))
print("Rest review/allowed rows:", len(rest_review))
print("Data quality rows:", len(data_quality))

# Prepare service audit without Python list columns.
services_audit = services_df.drop(columns=[c for c in ["_net_50_intervals", "_net_breaks_intervals"] if c in services_df.columns]).copy()

# Export clean sheets.
sheets = {
    "SUMMARY_BY_MONTH": summary_by_month,
    "ALL_INFRACTIONS": all_infractions,
    "OVER_50H_WEEK": over50_detail,
    "STREAK_7DAYS": streak_detail,
    "SPAN_OVER_14H": span_detail,
    "REST_UNDER_11H": rest_detail,
    "REST_REVIEW_ALLOWED": rest_review,
    "PAUSE_INSUFF": breaks_detail,
    "PAUSE_AUDIT_SERVICES": breaks_audit,
    "SERVICES_AUDIT": services_audit,
    "CALENDAR_HOUR_SLICES": calendar_slices_df,
    "DATA_QUALITY": data_quality,
}

# Reasonable column ordering for key sheets.
ORDERS = {
    "ALL_INFRACTIONS": ["TARGET_MONTH", "EVENT_DATE", "RULE", "SEVERITY", "Collaborateur", "No_collaborateur_codes", "Match_status", "DETAIL", "service_id", "service_date", "service_start", "service_end"],
    "SERVICES_AUDIT": ["service_month", "service_date", "service_start", "service_end", "continues_after_midnight", "continuation_row_count", "creates_worked_day", "Collaborateur", "No_collaborateur_codes", "Match_status", "amplitude_hours", "net_50h_minutes", "net_breaks_minutes", "pause_minutes_inside_service", "attached_calendar_dates", "service_id"],
    "CALENDAR_HOUR_SLICES": ["target_month", "calendar_date", "week_monday", "slice_start", "slice_end", "minutes", "hours", "continuation_from_previous_service", "service_date", "Collaborateur", "No_collaborateur_codes", "service_id", "note"],
    "DATA_QUALITY": ["TARGET_MONTH", "EVENT_DATE", "QUALITY_TYPE", "DETAIL", "Collaborateur", "No collaborateur", "No prestation", "Prestation", "start_dt_local", "end_dt_local", "interval_status", "service_id", "service_date", "continuation_from_previous_service"],
}

with pd.ExcelWriter(MULTIPLE_BOOK_PATH, engine="openpyxl", mode="w") as writer:
    for sheet_name, df_sheet in sheets.items():
        clean = excel_safe_no_tz(prep_for_export(df_sheet, drop_collaborateur_id=True))
        clean = reorder_columns(clean, ORDERS.get(sheet_name, []))
        clean = ensure_unique_columns(clean)
        clean.to_excel(writer, index=False, sheet_name=sheet_name[:31])

apply_swiss_formats_xlsx(MULTIPLE_BOOK_PATH)

print("\n✅ HYBRID MULTI-MONTH WORKBOOK CREATED:")
print(MULTIPLE_BOOK_PATH)

print("\n===== IMPORTANT HYBRID LOGIC NOTE =====")
print("• 7-day streak, rest <11h, span >14h, and break checks use service_date = real service/shift start date.")
print("• Early-morning continuation rows and pause rows after midnight are attached to the previous service when they overlap it.")
print("• They do NOT create a fake worked day.")
print("• 50h/week uses calendar slices, so hours are allocated to the actual date/week where they physically happened.")

try:
    display(summary_by_month)
except Exception:
    pass


===== LOAD INPUTS =====
✅ Excluded prestations {'60041', '196'}: 11696 -> 11631 rows
✅ Loaded rows: 11590
Match status counts:
collab_match_status
OK_FALLBACK    11590
Name: count, dtype: int64

Interval status counts:
interval_status
OK    11590
Name: count, dtype: int64

===== BUILD HYBRID SERVICES =====
  ...service groups processed: 50/77
✅ Services built: 1039
✅ Orphan pause rows: 0
✅ Services built: 1039
✅ Calendar hour slices: 2088
✅ Orphan pauses not counted as workdays: 0
Service months: ['2026-04']
Calendar months with hours: ['2026-04']

===== RUN CHECKS =====
Over 50h weeks: 1
7-day streaks: 0
Span >14h services: 8
Rest infractions: 19
Break infractions: 40
Total hard/potential infractions: 68
Rest review/allowed rows: 63
Data quality rows: 0
✅ Swiss-FR formats applied: /content/drive/MyDrive/ltr check all UO/multiple/LTR_CHECKS_MULTIPLE_ALL_MONTHS_HYBRID.xlsx

✅ HYBRID MULTI-MONTH WORKBOOK CREATED:
/content/drive/MyDrive/ltr check all UO/multiple/LTR_CHECKS_MULTIPLE_ALL_M

,TARGET_MONTH,OVER_50H_WEEK,STREAK_7DAYS,SPAN_OVER_14H,REST_UNDER_11H,PAUSE_INSUFF,TOTAL_INFRACTIONS,SERVICES_STARTED,CALENDAR_HOURS,ORPHAN_PAUSE_ROWS,INVALID_INTERVAL_ROWS
0,2026-04,1,0,8,19,40,68,1039,5280.0,0,0


In [11]:
#@title 11) Check for Unrecognized Collaborators

print("=== UNRECOGNIZED COLLABORATORS CHECK ===")

# In the script, unmapped collabs receive a key starting with 'UNMATCHED_CODE::' or 'UNMATCHED_NAME::'
unmatched_mask = df['collab_key'].astype(str).str.startswith('UNMATCHED')

# We should also catch ambiguous matches (where the same code maps to multiple different master IDs)
ambig_mask = df['collab_match_status'] == 'AMBIG_MATCH'

# Filter the dataframe for these issues
unmatched_df = df[unmatched_mask | ambig_mask].copy()

if unmatched_df.empty:
    print("✅ All RDA rows were successfully recognized and mapped to a known collaborator!")
else:
    print(f"⚠️ Found {len(unmatched_df)} rows with recognition issues (new/missing codes or ambiguous mapping).")

    # Group by the raw input columns to show exactly which codes are failing
    summary = unmatched_df.groupby(
        ['Collaborateur', 'No collaborateur', 'collab_match_status', 'collab_key'],
        dropna=False
    ).size().reset_index(name='Row Count')

    # Sort for better readability
    summary = summary.sort_values('Row Count', ascending=False).reset_index(drop=True)
    display(summary)


=== UNRECOGNIZED COLLABORATORS CHECK ===
✅ All RDA rows were successfully recognized and mapped to a known collaborator!
